In [7]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from polars import Null

from func.UsefulFunctions import *

This notebook documents the cleaning and inspection process for the artist data on wikidata

In [8]:
artistDF = pl.read_parquet("./Data/pipelinesArtistsWikidata.parquet")
print(f"There are {len(artistDF["artist"].unique())} unique artist entities after the removal of the disallowed occupations.")
display(artistDF.head(3))

There are 444888 unique artist entities after the removal of the disallowed occupations.


artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""Q7161""","""Q36834""","""Q6581097""","""composer""","""male""","""Q29""",null,null,null,"""Q1413570""","""https://en.wikipedia.org/wiki/…","""533761c0-61cd-4d0e-b264-c71d5d…",null,null,"""Spain""","""sardana""","""José María Ventura Casas"""
"""Q7241""","""Q36834""","""Q6581097""","""composer""","""male""","""Q129286""",null,null,538.0,null,"""https://en.wikipedia.org/wiki/…","""13274b60-181b-431a-b049-b48560…",null,null,"""British Raj""",null,"""Rabindranath Tagore"""
"""Q7241""","""Q36834""","""Q6581097""","""composer""","""male""","""Q129286""",null,null,538.0,null,"""https://en.wikipedia.org/wiki/…","""13274b60-181b-431a-b049-b48560…",null,null,"""British Raj""",null,"""Rabindranath Tagore"""


In [19]:
#inspect the different fields in the system that are error prone
#see the different genders in the dataframe
print(artistDF[["artist","genderLabel"]].unique().select(pl.col("genderLabel").value_counts(sort=True)).unnest("genderLabel"))
print(artistDF[["genderLabel"]].unique().select(pl.col("genderLabel")).to_series().to_list())

#see some genders who have unknown or none gender
noGender = artistDF.filter(
    pl.col("gender").is_null()
)
noGender = noGender.select(pl.col(["artist","gender","genderLabel"])).unique()
print(noGender)
#Save the file for individuals who do not have a gender
noGender.write_csv("./missingValuesStore/artistsEntitiesWithoutGenders.csv",)


shape: (30, 2)
┌──────────────────┬────────┐
│ genderLabel      ┆ count  │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ male             ┆ 331681 │
│ female           ┆ 112112 │
│ non-binary       ┆ 472    │
│ trans woman      ┆ 471    │
│ trans man        ┆ 107    │
│ …                ┆ …      │
│ neutral sex      ┆ 1      │
│ apagender        ┆ 1      │
│ demiboy          ┆ 1      │
│ non-binary woman ┆ 1      │
│ demigirl         ┆ 1      │
└──────────────────┴────────┘
[None, 'intersex man', 'intersex trans woman', 'transgender', 'cisgender man', 'transmasculine', 'travesti', 'gender agnostic', 'two-spirit', 'non-binary', 'neutral sex', 'female', 'demigirl', 'genderfluid', 'intersex woman', 'genderqueer', 'trans man', 'demiboy', 'transfeminine', 'apagender', 'male', 'bigender', 'intersex', 'undisclosed gender', 'non-binary woman', 'takatāpui', 'trans woman', 'cisgender woman', 'agender', 'androgyne']
shape: (41, 3)
┌────────────┬──────

Checked the genders to ensure that error/unknown genders were successfully nulled.

In [23]:
#Null the artist labels for the individuals who do not have an artist label

noNames = artistDF.filter(
    pl.col("artist").is_in(pl.col("artistLabel").implode())
).select(pl.col("artistLabel")).unique()
name_dict = {k: None for k in noNames["artistLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("artistLabel").replace(name_dict))

noNamesDF = artistDF.filter(
    pl.col("artistLabel").is_null(),
).select(pl.col(["artist","artistLabel"])).unique()
display(noNamesDF)
print(f"There are {len(noNamesDF)} artists with no default labels/labels in english.")
noNamesDF.write_csv("./missingValuesStore/artistsEntitiesWithoutLabels.csv")

artist,artistLabel
str,str
"""Q20607958""",null
"""Q28355033""",null
"""Q6911564""",null
"""Q24856141""",null
"""Q134083012""",null
…,…
"""Q16722689""",null
"""Q105298015""",null
"""Q9119832""",null


There are 14131 artists with no default labels/labels in english.


In [32]:
#The individuals with a country without an english label
#First let's see if there are any unknown or erroneous countries by using the Q-numbers and there are
artistDF.filter(
    pl.col('countryCitizenshipLabel').str.starts_with("Q")
).select(pl.col('countryCitizenshipLabel')).unique().to_series().to_list()

noLabel_Country = artistDF.filter(
    pl.col("countryCitizenship").is_in(pl.col("countryCitizenshipLabel").implode())
).select(pl.col("countryCitizenshipLabel")).unique()
countryName_dict = {k: None for k in noLabel_Country["countryCitizenshipLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("countryCitizenshipLabel").replace(countryName_dict))
artistDF = artistDF.with_columns(
    countryCitizenship = pl.when(~pl.col('countryCitizenship').str.starts_with("Q")).then(None).otherwise(pl.col('countryCitizenship'))
)
noCountryLabel = artistDF.filter(
    pl.col("countryCitizenshipLabel").is_null(),
).select(pl.col(["countryCitizenship","countryCitizenshipLabel"])).unique()
display(noCountryLabel)
# noCountry = artistDF.filter(
#     pl.col("countryCitizenship").is_in(pl.col("countryCitizenshipLabel").implode())
# ).select(pl.col("countryCitizenship")).unique()
# display(noCountry)
# print(f"There are {len(noCountry)} artists with a country with no labels. These will be cleaned out.")
# noCountry.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledCountries.csv")

countryCitizenship,countryCitizenshipLabel
str,str
"""Q20581158""",null
null,null
"""Q2181059""",null


In [81]:
#Genres with no labels in english/default language
noGenre = artistDF.filter(
    pl.col("genre").is_in(pl.col("genreLabel").implode())
).select(pl.col(["artist",'genre'])).unique()
display(noGenre)
print(f"There are {len(noGenre)} artists with a genre with no labels. These will be cleaned out.")
noGenre.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledGenres.csv")

artist,genre
str,str


There are 0 artists with a genre with no labels. These will be cleaned out.


In [82]:
#Record Labels without a label
noRecordLabelLabel = artistDF.filter(
    pl.col("recordLabel").is_in(pl.col("recordLabelLabel").implode())
).select(pl.col(["artist","recordLabel"])).unique()
display(noRecordLabelLabel)
print(f"There are {len(noRecordLabelLabel)} artists with a genre with no labels. These will be cleaned out.")
noRecordLabelLabel.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledRecordLabels.csv")

artist,recordLabel
str,str
"""Q108475098""","""Q115781660"""
"""Q494596""","""Q113455672"""
"""Q13026009""","""Q13026701"""
"""Q16140577""","""Q13026701"""
"""Q106464771""","""Q106576703"""
…,…
"""Q12880672""","""Q133445441"""
"""Q359835""","""Q2100905"""
"""Q11367442""","""Q9081271"""


There are 198 artists with a genre with no labels. These will be cleaned out.


In [83]:
#Influences Labels without a label
noLabelsInfluencedBY = artistDF.filter(
    pl.col("influencedBY").is_in(pl.col("influencedBYLabel").implode(), nulls_equal=True)
).select(pl.col(["artist", "influencedBY"])).unique()
display(noLabelsInfluencedBY)
print(f"There are {len(noLabelsInfluencedBY)} artists with an influence with no labels. These will be cleaned out.")
noLabelsInfluencedBY.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledInfluences.csv")

artist,influencedBY
str,str
"""Q139426544""",null
"""Q11681474""",null
"""Q94756156""",null
"""Q80834688""",null
"""Q4920541""",null
…,…
"""Q17327089""",null
"""Q4399376""",null
"""Q16399215""",null


There are 443571 artists with an influence with no labels. These will be cleaned out.


In [84]:
cleanedArtistDF = artistDF.filter(
    ~pl.col("artist").is_in(pl.col("artistLabel").implode()),
    ~pl.col("influencedBY").is_in(pl.col("influencedBYLabel").implode()),
    ~pl.col("recordLabel").is_in(pl.col("recordLabelLabel").implode()),
    ~pl.col("genre").is_in(pl.col("genreLabel").implode()),
)

#Add a column for presence of wikiPage
cleanedArtistDF = cleanedArtistDF.with_columns(
    isThereAWiki = pl.col("article").is_not_null()
)
#Finally de-duplicate the data.
cleanedArtistDF = cleanedArtistDF.unique()
display(cleanedArtistDF.head(5))
display(cleanedArtistDF.shape)

artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel,isThereAWiki
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,bool
"""Q194220""","""Q36834""","""Q6581097""","""composer""","""male""","""Q30""","""Q364214""","""Rakim""",null,"""Q753679""","""https://en.wikipedia.org/wiki/…","""cfbc0924-0035-4d6c-8197-f02465…","""Q183387""","""Columbia Records""","""United States""","""gangsta rap""","""Nas""",true
"""Q10681""","""Q36834""","""Q6581097""","""composer""","""male""","""Q142""","""Q3176402""","""Jef Le Penven""",222.0,"""Q910606""","""https://en.wikipedia.org/wiki/…","""abc61045-7ae3-48d9-91f2-d44689…","""Q21077""","""Warner Music Group""","""France""","""music of Brittany""","""Alan Stivell""",true
"""Q15123969""","""Q488205""","""Q48270""","""singer-songwriter""","""non-binary""","""Q145""","""Q182725""","""Etta James""",192.0,"""Q37073""","""https://en.wikipedia.org/wiki/…","""5a85c140-dcf9-4dd2-b2c8-aff047…","""Q193023""","""Capitol Records""","""United Kingdom""","""pop music""","""Sam Smith""",true
"""Q44479972""","""Q753110""","""Q6581097""","""songwriter""","""male""","""Q30""","""Q6107""","""Tupac Shakur""",null,"""Q15697534""","""https://en.wikipedia.org/wiki/…","""2282c539-05a2-477c-82b7-fe4079…","""Q59210322""","""Scumgang Records""","""United States""","""drill""","""6ix9ine""",true
"""Q230454""","""Q639669""","""Q6581072""","""musician""","""female""","""Q16""","""Q272913""","""Kitty Wells""",256.0,"""Q37073""","""https://en.wikipedia.org/wiki/…","""bf4bb21e-759f-4b22-bfd4-174593…","""Q843402""","""Warner Bros. Records""","""Canada""","""pop music""","""k.d. lang""",true


(136735, 18)

That is the end of the cleaning stage. Now time to analyse the data. Null values were left intentionally as we are looking for the gaps

In [85]:
#Discover the gender gap in the set.
artistGender = cleanedArtistDF.select(
    pl.col("artist"),
    pl.col("gender"),
    pl.col("genderLabel"),
)
artistGender = artistGender.unique()
artistGenderGrouped = artistGender.group_by(["gender", "genderLabel"]).agg(pl.col("artist").count())
artistGenderGroupedFRAC = artistGender["genderLabel"].value_counts(sort=True,normalize=True,name="fraction")
display(artistGenderGrouped)
display(artistGenderGroupedFRAC)

gender,genderLabel,artist
str,str,u32
"""Q6581097""","""male""",303
"""Q18116794""","""genderfluid""",4
"""Q1052281""","""trans woman""",1
"""Q6581072""","""female""",188
"""Q48270""","""non-binary""",4


genderLabel,fraction
str,f64
"""male""",0.606
"""female""",0.376
"""non-binary""",0.008
"""genderfluid""",0.008
"""trans woman""",0.002


In [86]:
genderChart = (
    alt.Chart(artistGenderGrouped).mark_bar().encode(
    y="artist",
    x="genderLabel",
    )
    .properties(width=500, title="Gender Distribution of Wikidata")
)
genderChart.encoding.x.title = "Genders"
genderChart.encoding.y.title = "Counts"
genderChart.show()



genderBase = alt.Chart(artistGenderGrouped).encode(
    alt.Theta("artist:Q").stack(True),
    alt.Color("genderLabel:N").legend(None)
    )
pie = genderBase.mark_arc(outerRadius=120)
text = genderBase.mark_text(radius=140, size=20).encode(text="genderLabel:N")
pie+text

alt.Chart(...)

alt.LayerChart(...)

In [87]:
#Top regions and region gap

regions = cleanedArtistDF.select(
    pl.col("artist"),
    pl.col("countryCitizenship"),
    pl.col("countryCitizenshipLabel"),
)

regions = regions.unique()
regionsGrouped = regions.group_by(["countryCitizenship", "countryCitizenshipLabel"]).agg(pl.col("artist").count())
display(regionsGrouped.head(5))

countryCitizenship,countryCitizenshipLabel,artist
str,str,u32
"""Q41""","""Greece""",5
"""Q159""","""Russia""",3
"""Q948""","""Tunisia""",1
"""Q884""","""South Korea""",1
"""Q33""","""Finland""",1


In [88]:
#Top genres and genre gap

In [89]:
#Top occupations

In [90]:
#Top occupations by gender

In [91]:
#Gender and genre split